In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max
import time

# Stop existing Spark session if needed
SparkSession.builder.getOrCreate().stop()

# Start new Spark session
spark = SparkSession.builder\
        .master("spark://192.168.2.213:7077") \
        .appName("test_cluster")\
        .config("spark.driver.host", "192.168.2.107") \
        .config("spark.driver.port", "4041") \
        .config("spark.ui.port", "4042") \
        .config("spark.dynamicAllocation.enabled", True)\
        .config("spark.dynamicAllocation.shuffleTracking.enabled", True)\
        .config("spark.shuffle.service.enabled", False)\
        .config("spark.dynamicAllocation.executorIdleTimeout", "30s")\
        .config("spark.executor.cores", 2)\
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("hdfs://192.168.2.213:9000/data/merged_data.csv", header=True, inferSchema=True)
df.describe()

DataFrame[summary: string, Recip_County: string, Recip_State: string, FIPS: string, MMWR_week: string, Completeness_pct: string, Administered_Dose1_Recip: string, Administered_Dose1_Pop_Pct: string, Administered_Dose1_Recip_5Plus: string, Administered_Dose1_Recip_5PlusPop_Pct: string, Administered_Dose1_Recip_12Plus: string, Administered_Dose1_Recip_12PlusPop_Pct: string, Administered_Dose1_Recip_18Plus: string, Administered_Dose1_Recip_18PlusPop_Pct: string, Administered_Dose1_Recip_65Plus: string, Administered_Dose1_Recip_65PlusPop_Pct: string, Series_Complete_Yes: string, Series_Complete_Pop_Pct: string, Series_Complete_5Plus: string, Series_Complete_5PlusPop_Pct: string, Series_Complete_5to17: string, Series_Complete_5to17Pop_Pct: string, Series_Complete_12Plus: string, Series_Complete_12PlusPop_Pct: string, Series_Complete_18Plus: string, Series_Complete_18PlusPop_Pct: string, Series_Complete_65Plus: string, Series_Complete_65PlusPop_Pct: string, Booster_Doses: string, Booster_Dos

In [13]:
df.describe()

DataFrame[summary: string, Recip_County: string, Recip_State: string, FIPS: string, MMWR_week: string, Completeness_pct: string, Administered_Dose1_Recip: string, Administered_Dose1_Pop_Pct: string, Administered_Dose1_Recip_5Plus: string, Administered_Dose1_Recip_5PlusPop_Pct: string, Administered_Dose1_Recip_12Plus: string, Administered_Dose1_Recip_12PlusPop_Pct: string, Administered_Dose1_Recip_18Plus: string, Administered_Dose1_Recip_18PlusPop_Pct: string, Administered_Dose1_Recip_65Plus: string, Administered_Dose1_Recip_65PlusPop_Pct: string, Series_Complete_Yes: string, Series_Complete_Pop_Pct: string, Series_Complete_5Plus: string, Series_Complete_5PlusPop_Pct: string, Series_Complete_5to17: string, Series_Complete_5to17Pop_Pct: string, Series_Complete_12Plus: string, Series_Complete_12PlusPop_Pct: string, Series_Complete_18Plus: string, Series_Complete_18PlusPop_Pct: string, Series_Complete_65Plus: string, Series_Complete_65PlusPop_Pct: string, Booster_Doses: string, Booster_Dos

In [14]:
df.printSchema()

# Show summary statistics for numerical columns
df.describe().show()

# Example: Correlation between administered doses and completed series
from pyspark.sql.functions import corr

# Replace these column names with the appropriate ones from your dataset
col1, col2 = "Administered_Dose1_Recip", "Series_Complete_Yes"
df.select(corr(col1, col2)).show()

# Show number of unique states and counties
df.select("Recip_State", "Recip_County").distinct().show()

# Group by state and get the average series completion percentage
df.groupBy("Recip_State").agg({"Series_Complete_Pop_Pct": "avg"}).show()

# Filter data for specific conditions (e.g., states with completion percentage > 80%)
df.filter(df["Series_Complete_Pop_Pct"] > 80).show()

# Count of records per MMWR week
df.groupBy("MMWR_week").count().show()

# Example of additional analysis: Calculate the percentage change in booster doses for a specific state
from pyspark.sql import functions as F

df_with_pct_change = df.withColumn(
    "Booster_Pct_Change",
    (df["Booster_Doses_Vax_Pct"] - df["Booster_Doses_5Plus_Vax_Pct"]) / df["Booster_Doses_5Plus_Vax_Pct"] * 100
)

df_with_pct_change.select("Recip_State", "Booster_Pct_Change").show()


root
 |-- Date: date (nullable = true)
 |-- Recip_County: string (nullable = true)
 |-- Recip_State: string (nullable = true)
 |-- FIPS: string (nullable = true)
 |-- MMWR_week: integer (nullable = true)
 |-- Completeness_pct: double (nullable = true)
 |-- Administered_Dose1_Recip: integer (nullable = true)
 |-- Administered_Dose1_Pop_Pct: double (nullable = true)
 |-- Administered_Dose1_Recip_5Plus: integer (nullable = true)
 |-- Administered_Dose1_Recip_5PlusPop_Pct: double (nullable = true)
 |-- Administered_Dose1_Recip_12Plus: integer (nullable = true)
 |-- Administered_Dose1_Recip_12PlusPop_Pct: double (nullable = true)
 |-- Administered_Dose1_Recip_18Plus: integer (nullable = true)
 |-- Administered_Dose1_Recip_18PlusPop_Pct: double (nullable = true)
 |-- Administered_Dose1_Recip_65Plus: integer (nullable = true)
 |-- Administered_Dose1_Recip_65PlusPop_Pct: double (nullable = true)
 |-- Series_Complete_Yes: integer (nullable = true)
 |-- Series_Complete_Pop_Pct: double (nullable 

+-------+------------+-----------+------------------+------------------+-----------------+------------------------+--------------------------+------------------------------+-------------------------------------+-------------------------------+--------------------------------------+-------------------------------+--------------------------------------+-------------------------------+--------------------------------------+-------------------+-----------------------+---------------------+----------------------------+---------------------+----------------------------+----------------------+-----------------------------+----------------------+-----------------------------+----------------------+-----------------------------+-----------------+---------------------+-------------------+---------------------------+--------------------+----------------------------+--------------------+----------------------------+--------------------+----------------------------+--------------------+------------

+---------------------------------------------------+
|corr(Administered_Dose1_Recip, Series_Complete_Yes)|
+---------------------------------------------------+
|                                 0.9920893739892219|
+---------------------------------------------------+



+-----------+--------------+
|Recip_State|  Recip_County|
+-----------+--------------+
|         KS|        Ottawa|
|         GA|      Muscogee|
|         IA|         Adair|
|         PR|       Guanica|
|         NE|         Adams|
|         ND|        Pierce|
|         SC|     Allendale|
|         NM|    Bernalillo|
|         TX|        Blanco|
|         MN|         Brown|
|         NH|       Unknown|
|         AR|        Saline|
|         TN|      Grainger|
|         MS|         Stone|
|         IL|      Hamilton|
|         MS|         Hinds|
|         MO|      Harrison|
|         LA|East Feliciana|
|         FL|      Hernando|
|         MN|        Isanti|
+-----------+--------------+
only showing top 20 rows



+-----------+----------------------------+
|Recip_State|avg(Series_Complete_Pop_Pct)|
+-----------+----------------------------+
|         AZ|          43.490387267904495|
|         SC|           34.80025747389499|
|         LA|           32.37379605959669|
|         MN|           41.93460424526507|
|         NJ|           47.18326289095519|
|         DC|          49.346321070234126|
|         OR|          42.619493221726856|
|         VA|           29.04921544583197|
|         RI|            47.4501015965167|
|         KY|          34.685990639193534|
|         WY|          31.875656274192405|
|         NH|           42.95672105672106|
|         MI|           40.64723497704126|
|         NV|          31.685707560493352|
|         WI|          42.731981981981974|
|         ID|          32.302630104232804|
|         CA|           39.56718892383668|
|         NE|          30.026005457561237|
|         CT|           50.22030921931667|
|         MT|           34.49023068084727|
+----------

+----------+-------------+-----------+-----+---------+----------------+------------------------+--------------------------+------------------------------+-------------------------------------+-------------------------------+--------------------------------------+-------------------------------+--------------------------------------+-------------------------------+--------------------------------------+-------------------+-----------------------+---------------------+----------------------------+---------------------+----------------------------+----------------------+-----------------------------+----------------------+-----------------------------+----------------------+-----------------------------+-------------+---------------------+-------------------+---------------------------+--------------------+----------------------------+--------------------+----------------------------+--------------------+----------------------------+--------------------+----------------------------+------

[Stage 15:=================================================>        (6 + 1) / 7]

+---------+-----+
|MMWR_week|count|
+---------+-----+
|       31|26122|
|       53|22855|
|       34|26122|
|       28|26122|
|       26|26122|
|       27|26122|
|       44|26136|
|       12|48988|
|       22|45724|
|       47|26136|
|        1|52258|
|       52|48986|
|       13|48984|
|        6|48991|
|       16|48991|
|        3|45724|
|       20|45724|
|       40|26122|
|       48|26136|
|        5|48991|
+---------+-----+
only showing top 20 rows

+-----------+------------------+
|Recip_State|Booster_Pct_Change|
+-----------+------------------+
|         IA|              NULL|
|         ID|              NULL|
|         WI|              NULL|
|         OK|              NULL|
|         MD|              NULL|
|         VA|              NULL|
|         KY|              NULL|
|         TX|              NULL|
|         AZ|              NULL|
|         AR|              NULL|
|         IA|              NULL|
|         AL|              NULL|
|         ID|              NULL|
|         PR| 

In [15]:
from pyspark.sql.functions import mean

# Define threshold for high and low vaccination (e.g., 60% as cut-off)
threshold = 60  

high_vax = df.filter(df.Series_Complete_Pop_Pct >= threshold)
low_vax = df.filter(df.Series_Complete_Pop_Pct < threshold)

# Compute average deaths per group
high_vax.select(mean("deaths").alias("Avg_Deaths_High_Vax")).show()
low_vax.select(mean("deaths").alias("Avg_Deaths_Low_Vax")).show()


+-------------------+
|Avg_Deaths_High_Vax|
+-------------------+
|  690.9611178980435|
+-------------------+



[Stage 22:=================================================>        (6 + 1) / 7]

+------------------+
|Avg_Deaths_Low_Vax|
+------------------+
|169.93599200247564|
+------------------+



In [16]:
from pyspark.sql.functions import corr

# Compute correlation between full vaccination rate and deaths
df.select(corr("Series_Complete_Pop_Pct", "deaths")).show()


[Stage 25:=================================================>        (6 + 1) / 7]

+-------------------------------------+
|corr(Series_Complete_Pop_Pct, deaths)|
+-------------------------------------+
|                   0.1575377800186528|
+-------------------------------------+

